In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import os
import matplotlib.ticker as ticker

In [ ]:
z_list = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
z_smol_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 1.5, 2.0, 3.01, 4.01, 5.0, 6.01, 8.01, 10.0]
snap_list = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]
snap_smol_list = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,8,4]
smol_id = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,15,17]

colors = plt.colormaps['viridis'](np.linspace(0, 1, 4))
colors[3] = [1.0, 0.65, 0.0, 1.0]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    'font.size': 12,
})

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (f"The voxel width is {dl} cMpc/h")

norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in z_list:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"{n} snapshots")

In [ ]:
id = 0

coords_z0 = il.snapshot.loadSubset(basePath, snap_list[id], 'dm', fields = ['Coordinates'])/1000 #in cMpc/h

z_min = 0.0
z_max = 12.5
mask = (coords_z0[:,2] > z_min) & (coords_z0[:,2] < z_max)

x = coords_z0[:, 0][mask]
y = coords_z0[:, 1][mask]
z = coords_z0[:, 2][mask]

fig = plt.figure()

ax2 = fig.add_subplot(111)
counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 256, cmap = "berlin_r", norm = LogNorm(), range=[[0,35], [0,35]])
#ax2.set_xlabel("x [cMpc/h]")
#ax2.set_ylabel("y [cMpc/h]")
ax2.set_aspect('equal')
#ax2.yaxis.set_major_locator(ticker.MultipleLocator(5))
#ax2.xaxis.set_major_locator(ticker.MultipleLocator(5))
#fig.colorbar(im2, ax = ax2, label = "Particle Counts")

ax2.set_xticks([])
ax2.set_yticks([])

plt.tight_layout()

plt.savefig('nbody_proj.png', transparent=True, bbox_inches="tight")
plt.savefig('nbody_proj.pdf', transparent=True, bbox_inches="tight");

In [ ]:
id = 0


coords_z0 = il.snapshot.loadSubset(basePath, snap_list[id], 'dm', fields = ['Coordinates'])/1000 #in cMpc/h

slice_index = 67

z_min = slice_index * dl
z_max = (slice_index + 1) * dl
mask = (coords_z0[:,2] > z_min) & (coords_z0[:,2] < z_max)

x = coords_z0[:, 0][mask]
y = coords_z0[:, 1][mask]
z = coords_z0[:, 2][mask]

fig = plt.figure(figsize=(11,4))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(np.log10(norm_dens[id][:,:,slice_index]),origin = "lower",cmap="inferno", extent = [0,35,0,35])
ax1.set_xlabel("x [cMpc/h]")
ax1.set_ylabel("y [cMpc/h]")
ax1.set_aspect('equal')
ax1.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
fig.colorbar(im1, ax = ax1, label = r"$\log (1+\delta)$")

ax2 = fig.add_subplot(1,2,2)
counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 256, cmap = "inferno", norm = LogNorm(), range=[[0,35], [0,35]])
ax2.set_xlabel("x [cMpc/h]")
ax2.set_ylabel("y [cMpc/h]")
ax2.set_aspect('equal')
ax2.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax2.xaxis.set_major_locator(ticker.MultipleLocator(5))
fig.colorbar(im2, ax = ax2, label = "Particle Counts")

plt.tight_layout()

plt.savefig('ps_dtfe_vs_nbody.png', transparent=True, bbox_inches="tight")
plt.savefig('ps_dtfe_vs_nbody.pdf', transparent=True, bbox_inches="tight");

In [ ]:
id = 0

slice = 95
from matplotlib.lines import Line2D
fig = plt.figure()
ax = fig.add_subplot(111)

im = ax.imshow(np.log10(norm_dens[id][:,:,slice]), origin="lower", cmap= "gray", extent = [0,35,0,35])

ax.contour(walls[id][:,:,slice], colors = colors[2], extent = [0,35,0,35], linewidths=.31, antialiased=True)
ax.contour(filaments[id][:,:,slice], colors = colors[1], extent = [0,35,0,35], linewidths=.31, antialiased=True)
ax.contour(nodes[id][:,:,slice], colors = colors[0], extent = [0,35,0,35], linewidths=.31, antialiased=True)

legend_elements = [
    Line2D([0], [0], color=colors[0], lw=2, label='Nodes'),
    Line2D([0], [0], color=colors[1], lw=2, label='Filaments'),
    Line2D([0], [0], color=colors[2], lw=2, label='Walls')
]
ax.legend(handles=legend_elements, loc="upper right", 
          facecolor="black", labelcolor="white", framealpha=0.8, edgecolor="none")

fig.colorbar(im, label = r"$\log (1+\delta)$")
ax.set_xlabel("x [cMpc/h]")
ax.set_ylabel("y [cMpc/h]")

plt.savefig("nexus_demo.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_demo.pdf", transparent = True, bbox_inches="tight");

In [ ]:
v_z0 = voids[0] * norm_dens[0]
f_z0 = filaments[0] * norm_dens[0]
w_z0 = walls[0] * norm_dens[0]
n_z0 = nodes[0] * norm_dens[0]

In [ ]:
plt.hist(np.log10(v_z0.ravel()[np.flatnonzero(v_z0)]), bins = "scott", color = colors[3], density=True, histtype = "step",label = "Voids", linewidth = 2)
plt.hist(np.log10(w_z0.ravel()[np.flatnonzero(w_z0)]), bins = "scott", color = colors[2], density=True, histtype = "step",label = "Walls", linewidth = 2)
plt.hist(np.log10(f_z0.ravel()[np.flatnonzero(f_z0)]), bins = "scott", color = colors[1], density=True, histtype = "step",label = "Filaments", linewidth = 2)
plt.hist(np.log10(n_z0.ravel()[np.flatnonzero(n_z0)]), bins = "scott", color = colors[0], density=True, histtype = "step",label = "Nodes", linewidth = 2)
plt.hist(np.log10(norm_dens[0].ravel()), bins="scott", color="grey", density=True, histtype="step", label="Total", linewidth=1)
plt.ylabel("PDF")
plt.grid()
plt.legend()
plt.xlabel(r"$\log (1+\delta)$")

plt.savefig("nexus_hist.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_hist.pdf", transparent = True, bbox_inches="tight");

In [ ]:
id = 0
slice = 95
from matplotlib.lines import Line2D
fig = plt.figure(figsize=(12,4))
ax1 = fig.add_subplot(121)

im = ax1.imshow(np.log10(norm_dens[id][:,:,slice]), origin="lower", cmap= "inferno", extent = [0,35,0,35])

ax1.contour(walls[id][:,:,slice], colors = colors[2], extent = [0,35,0,35], linewidths=.31, antialiased=True)
ax1.contour(filaments[id][:,:,slice], colors = colors[1], extent = [0,35,0,35], linewidths=.31, antialiased=True)
ax1.contour(nodes[id][:,:,slice], colors = colors[0], extent = [0,35,0,35], linewidths=.31, antialiased=True)

legend_elements = [
    Line2D([0], [0], color=colors[0], lw=2, label='Nodes'),
    Line2D([0], [0], color=colors[1], lw=2, label='Filaments'),
    Line2D([0], [0], color=colors[2], lw=2, label='Walls')
]
ax1.legend(handles=legend_elements, loc="upper right", 
          facecolor="black", labelcolor="white", framealpha=0.8, edgecolor="none")

fig.colorbar(im, label = r"$\log (1+\delta)$")
ax1.set_xlabel("x [cMpc/h]")
ax1.set_ylabel("y [cMpc/h]")

ax2 = fig.add_subplot(122)
ax2.hist(np.log10(v_z0.ravel()[np.flatnonzero(v_z0)]), bins = "scott", color = colors[3], density=True, histtype = "step",label = "Voids", linewidth = 2)
ax2.hist(np.log10(w_z0.ravel()[np.flatnonzero(w_z0)]), bins = "scott", color = colors[2], density=True, histtype = "step",label = "Walls", linewidth = 2)
ax2.hist(np.log10(f_z0.ravel()[np.flatnonzero(f_z0)]), bins = "scott", color = colors[1], density=True, histtype = "step",label = "Filaments", linewidth = 2)
ax2.hist(np.log10(n_z0.ravel()[np.flatnonzero(n_z0)]), bins = "scott", color = colors[0], density=True, histtype = "step",label = "Nodes", linewidth = 2)
ax2.hist(np.log10(norm_dens[0].ravel()[norm_dens[0].ravel() > 0]), bins="scott", color="grey", density=True, histtype="step", label="Total", linewidth=2)
ax2.grid()
ax2.set_ylabel("PDF")
ax2.legend()
ax2.set_xlabel(r"$\log (1+\delta)$")

plt.savefig('nexus_trial.png', transparent=True, bbox_inches="tight")
plt.savefig('nexus_trial.pdf', transparent=True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(8, 8)) 

maxmax = np.max(np.log10(norm_dens))
minmin = np.min(np.log10(norm_dens))

for i in range(16):
    ax = fig.add_subplot(4, 4, 16-i)
    ax.imshow(np.log10(norm_dens[smol_id[i]][:,:,13]), origin="lower", cmap="inferno", vmax = maxmax, vmin = minmin)
    
    ax.set_xticks([])
    ax.set_yticks([])
  
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)  
        spine.set_color('white')

    ax.text(
        0.04, 0.96,                  # X and Y position
        f"$z = {z_smol_list[i]}$",   # The text to display
        transform=ax.transAxes,      # Use axis coordinates (0 to 1) instead of data coordinates
        fontsize=12, 
        color="black", 
        verticalalignment="top", 
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=5)
        )    
        
fig.subplots_adjust(wspace=0, hspace=0)

plt.savefig("density_evol.png", transparent = True, bbox_inches="tight")
plt.savefig("density_evol.pdf", transparent = True, bbox_inches="tight");

In [ ]:
vol_n = []
vol_f = []
vol_w = []
vol_v = []

tot_vol = res ** 3

for i in range(n):
    print(i)
    vol_v.append(np.sum(voids[i].flat)/tot_vol)
    vol_w.append(np.sum(walls[i].flat)/tot_vol)
    vol_f.append(np.sum(filaments[i].flat)/tot_vol)
    vol_n.append(np.sum(nodes[i].flat)/tot_vol)

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(z_list, vol_n,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_list, vol_f,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_list, vol_w,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_list, vol_v,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Volume Fraction")
ax.legend()

plt.savefig("nexus_vol_frac.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_vol_frac.pdf", transparent = True, bbox_inches="tight");

In [ ]:
z_new = [0.0]

vol_n_new = [vol_n[0]]
vol_f_new = [vol_f[0]]
vol_w_new = [vol_w[0]]
vol_v_new = [vol_v[0]]

for i in range(1,n-1):
    print(i)

    vol_v_new.append((vol_v[i-1] + vol_v[i] + vol_v[i+1])/3)
    vol_w_new.append((vol_w[i-1] + vol_w[i] + vol_w[i+1])/3)
    vol_f_new.append((vol_f[i-1] + vol_f[i] + vol_f[i+1])/3)
    vol_n_new.append((vol_n[i-1] + vol_n[i] + vol_n[i+1])/3)

    z_new.append((z_list[i-1]+ z_list[i] + z_list[i+1])/3)

z_new.append(10.0)

vol_n_new.append(vol_n[17])
vol_f_new.append(vol_f[17])
vol_w_new.append(vol_w[17])
vol_v_new.append(vol_v[17])

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(z_new, vol_n_new,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_new, vol_f_new,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_new, vol_w_new,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_new, vol_v_new,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Volume Fraction")
ax.legend()

plt.savefig("nexus_vol_frac_new.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_vol_frac_new.pdf", transparent = True, bbox_inches="tight");

In [ ]:
mass_n = []
mass_f = []
mass_w = []
mass_v = []

tot_cells = res**3

for i in range(n):
    print(i)
    mass_v.append(np.sum((voids[i] * norm_dens[i]).flat)/tot_cells)
    mass_w.append(np.sum((walls[i] * norm_dens[i]).flat)/tot_cells)
    mass_f.append(np.sum((filaments[i] * norm_dens[i]).flat)/tot_cells)
    mass_n.append(np.sum((nodes[i] * norm_dens[i]).flat)/tot_cells)

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(z_list, mass_n,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_list, mass_f,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_list, mass_w,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_list, mass_v,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Mass Fraction")
ax.legend()

plt.savefig("nexus_mass_frac.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_mass_frac.pdf", transparent = True, bbox_inches="tight");

In [ ]:
z_new = [0.0]

mass_n_new = [mass_n[0]]
mass_f_new = [mass_f[0]]
mass_w_new = [mass_w[0]]
mass_v_new = [mass_v[0]]

for i in range(1,n-1):
    print(i)

    mass_v_new.append((mass_v[i-1] + mass_v[i] + mass_v[i+1])/3)
    mass_w_new.append((mass_w[i-1] + mass_w[i] + mass_w[i+1])/3)
    mass_f_new.append((mass_f[i-1] + mass_f[i] + mass_f[i+1])/3)
    mass_n_new.append((mass_n[i-1] + mass_n[i] + mass_n[i+1])/3)

    z_new.append((z_list[i-1]+ z_list[i] + z_list[i+1])/3)

z_new.append(10.0)

mass_n_new.append(mass_n[17])
mass_f_new.append(mass_f[17])
mass_w_new.append(mass_w[17])
mass_v_new.append(mass_v[17])

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(z_new, mass_n_new,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_new, mass_f_new,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_new, mass_w_new,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_new, mass_v_new,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Mass Fraction")
ax.legend()

plt.savefig("nexus_mass_frac_new.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_mass_frac_new.pdf", transparent = True, bbox_inches="tight");

In [ ]:
labels = 'Nodes', 'Filaments', 'Walls', 'Voids'
masses = [mass_n[0], mass_f[0], mass_w[0], mass_v[0]]
vols = [vol_n[0], vol_f[0], vol_w[0], vol_v[0]]


fig = plt.figure(figsize=(12,5))

ax = fig.add_subplot(121)
ax.pie(masses, labels=labels, autopct='%1.1f%%', colors=colors)
ax.set_title("The Cosmic Web by Mass")

ax = fig.add_subplot(122)
ax.pie(vols, labels=labels, autopct='%1.1f%%', colors=colors)
ax.set_title("The Cosmic Web by Volume")

plt.savefig("nexus_pie.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_pie.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(8, 8)) 

for i in range(16):
    total = nodes[smol_id[i]]*4 + filaments[smol_id[i]]*3 + walls[smol_id[i]]*2 + voids[smol_id[i]]
    ax = fig.add_subplot(4, 4, 16-i)
    ax.imshow(total[:,:,13], origin="lower", cmap="viridis_r", vmax = 4, vmin = 1)
    
    ax.set_xticks([])
    ax.set_yticks([])
  
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)  
        spine.set_color('white')
        
    ax.text(
        0.04, 0.96,                  # X and Y position
        f"$z = {z_smol_list[i]}$",   # The text to display
        transform=ax.transAxes,      # Use axis coordinates (0 to 1) instead of data coordinates
        fontsize=12, 
        color="white", 
        verticalalignment="top", 
        bbox=dict(facecolor="black", alpha=0.75, edgecolor="none", pad=5)
        )  
        
fig.subplots_adjust(wspace=0, hspace=0)

plt.savefig("nexus_evol_slice.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_evol_slice.pdf", transparent = True, bbox_inches="tight");

In [ ]:
history = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/history.npy")

In [ ]:
struct = ["Nodes","Filaments","Walls","Voids"]

fig = plt.figure(figsize=(12,8))

for i in range(4):

    ax = fig.add_subplot(2,2,i+1)

    ax.plot(z_list, history[i][0], label = "Nodes", marker = "x", c=colors[0])
    ax.plot(z_list, history[i][1], label = "Filaments", marker = "o", c=colors[1])
    ax.plot(z_list, history[i][2], label = "Walls", marker = "*", c=colors[2])
    ax.plot(z_list, history[i][3], label = "Voids", marker = "s", c=colors[3])

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Origin of {struct[i]} Particles")
    ax.legend()

plt.tight_layout()

plt.savefig("history.png", transparent = True, bbox_inches="tight")
plt.savefig("history.pdf", transparent = True, bbox_inches="tight");

In [ ]:
# 1. Create a master list to store the smoothed data for all 4 origins (j)
all_smoothed_histories = []

for j in range(4):
    # Initialize the smoothed lists with the first snapshot (index 0)
    hist_n = [history[j][0][0]]
    hist_f = [history[j][1][0]]
    hist_w = [history[j][2][0]]
    hist_v = [history[j][3][0]]

    # 2. Centered moving average (skipping the first and last points)
    for i in range(1, n-1):
        hist_n.append((history[j][0][i-1] + history[j][0][i] + history[j][0][i+1]) / 3)
        hist_f.append((history[j][1][i-1] + history[j][1][i] + history[j][1][i+1]) / 3)
        hist_w.append((history[j][2][i-1] + history[j][2][i] + history[j][2][i+1]) / 3)
        hist_v.append((history[j][3][i-1] + history[j][3][i] + history[j][3][i+1]) / 3)

    # 3. Append the final snapshot (using n-1 instead of a hardcoded 17)
    hist_n.append(history[j][0][n-1])
    hist_f.append(history[j][1][n-1])
    hist_w.append(history[j][2][n-1])
    hist_v.append(history[j][3][n-1])
    
    # 4. Save this specific j's smoothed data into the master list
    all_smoothed_histories.append([hist_n, hist_f, hist_w, hist_v])


# --- PLOTTING ---

fig = plt.figure(figsize=(12, 8))

for i in range(4):
    ax = fig.add_subplot(2, 2, i+1)

    # 5. Extract the 1D lists from our master container for the plot
    # all_smoothed_histories[i][0] corresponds to 'hist_n' for origin i
    ax.plot(z_new, all_smoothed_histories[i][0], label="Nodes", marker="x", c=colors[0])
    ax.plot(z_new, all_smoothed_histories[i][1], label="Filaments", marker="o", c=colors[1])
    ax.plot(z_new, all_smoothed_histories[i][2], label="Walls", marker="*", c=colors[2])
    ax.plot(z_new, all_smoothed_histories[i][3], label="Voids", marker="s", c=colors[3])

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Origin of {struct[i]} Particles")
    ax.legend()

plt.tight_layout()

plt.savefig("history_new.png", transparent=True, bbox_inches="tight")
plt.savefig("history_new.pdf", transparent=True, bbox_inches="tight");

In [ ]:
evolution = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/evolution.npy")

In [ ]:
fig = plt.figure(figsize=(6,12))

for i in range(1,4):

    ax = fig.add_subplot(3,1,i)

    ax.plot(z_list, evolution[i][0], label = "Nodes", marker = "x", c=colors[0])
    ax.plot(z_list, evolution[i][1], label = "Filaments", marker = "o", c=colors[1])
    ax.plot(z_list, evolution[i][2], label = "Walls", marker = "*", c=colors[2])
    ax.plot(z_list, evolution[i][3], label = "Voids", marker = "s", c=colors[3])

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Evolution of {struct[i]} Particles")
    ax.legend()

plt.tight_layout()

plt.savefig("evolution.png", transparent = True, bbox_inches="tight")
plt.savefig("evolution.pdf", transparent = True, bbox_inches="tight");

In [ ]:
# 1. Create a master list to store the smoothed data for all 4 origins (j)
all_smoothed_evolutions = []

for j in range(4):
    # Initialize the smoothed lists with the first snapshot (index 0)
    evol_n = [evolution[j][0][0]]
    evol_f = [evolution[j][1][0]]
    evol_w = [evolution[j][2][0]]
    evol_v = [evolution[j][3][0]]

    # 2. Centered moving average (skipping the first and last points)
    for i in range(1, n-1):
        evol_n.append((evolution[j][0][i-1] + evolution[j][0][i] + evolution[j][0][i+1]) / 3)
        evol_f.append((evolution[j][1][i-1] + evolution[j][1][i] + evolution[j][1][i+1]) / 3)
        evol_w.append((evolution[j][2][i-1] + evolution[j][2][i] + evolution[j][2][i+1]) / 3)
        evol_v.append((evolution[j][3][i-1] + evolution[j][3][i] + evolution[j][3][i+1]) / 3)

    # 3. Append the final snapshot (using n-1 instead of a hardcoded 17)
    evol_n.append(evolution[j][0][n-1])
    evol_f.append(evolution[j][1][n-1])
    evol_w.append(evolution[j][2][n-1])
    evol_v.append(evolution[j][3][n-1])
    
    # 4. Save this specific j's smoothed data into the master list
    all_smoothed_evolutions.append([evol_n, evol_f, evol_w, evol_v])


# --- PLOTTING ---

fig = plt.figure(figsize=(6, 12))

for i in range(1,4):
    ax = fig.add_subplot(3, 1, i)

    # 5. Extract the 1D lists from our master container for the plot
    # all_smoothed_histories[i][0] corresponds to 'hist_n' for origin i
    ax.plot(z_new, all_smoothed_evolutions[i][0], label="Nodes", marker="x", c=colors[0])
    ax.plot(z_new, all_smoothed_evolutions[i][1], label="Filaments", marker="o", c=colors[1])
    ax.plot(z_new, all_smoothed_evolutions[i][2], label="Walls", marker="*", c=colors[2])
    ax.plot(z_new, all_smoothed_evolutions[i][3], label="Voids", marker="s", c=colors[3])

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Evolution of {struct[i]} Particles")
    ax.legend()

plt.tight_layout()

plt.savefig("evolution_new.png", transparent=True, bbox_inches="tight")
plt.savefig("evolution_new.pdf", transparent=True, bbox_inches="tight");

In [ ]:
struct = ["Everything","Nodes","Filaments","Walls","Voids"]

cos_all = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_all.npy")
cos_avg = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_avg.npy")
cos_err_up = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_err_up.npy")
cos_err_down = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_err_down.npy")

mag_all = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_all.npy")
mag_avg = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_avg.npy")
mag_err_up = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_err_up.npy")
mag_err_down = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_err_down.npy")

diff_all = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_all.npy")
diff_avg = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_avg.npy")
diff_err_up = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_err_up.npy")
diff_err_down = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_err_down.npy")

In [ ]:
red = [ 20.05, 14.99, 11.98, 10.98, 10.00, 9.39, 9.00, 8.45, 8.01, 7.60, 7.24, 7.01, 6.49, 6.01,
    5.85, 5.53, 5.23, 5.00, 4.66, 4.43, 4.18, 4.01, 3.71, 3.49,
    3.28, 3.01, 2.90, 2.73, 2.58, 2.44, 2.32, 2.21, 2.10, 2.00,
    1.90, 1.82, 1.74, 1.67, 1.60, 1.53, 1.50, 1.41, 1.36, 1.30,
    1.25, 1.21, 1.15, 1.11, 1.07, 1.04, 1.00, 0.95, 0.92, 0.89,
    0.85, 0.82, 0.79, 0.76, 0.73, 0.70, 0.68, 0.64, 0.62, 0.60,
    0.58, 0.55, 0.52, 0.50, 0.48, 0.46, 0.44, 0.42, 0.40, 0.38,
    0.36, 0.35, 0.33, 0.31, 0.30, 0.27, 0.26, 0.24, 0.23, 0.21,
    0.20, 0.18, 0.17, 0.15, 0.14, 0.13, 0.11, 0.10, 0.08, 0.07,
    0.06, 0.05, 0.03, 0.02, 0.01, 0.00
]

In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(red, cos_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(red, (np.array(cos_avg[0])-np.array(cos_err_down[0])), 
                (np.array(cos_avg[0])+np.array(cos_err_up[0])), alpha=.2, color="black")
    
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(red[9:], cos_avg[k][9:], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("cos.png", transparent = True, bbox_inches="tight")
plt.savefig("cos.pdf", transparent = True, bbox_inches="tight");

In [ ]:
idd = -1
bin_type = "scott"
#bin_type = 100

plt.hist(cos_all[4][idd], bins = bin_type, color = colors[3], density=True, histtype = "step",label = "Voids", linewidth = 2)
plt.hist(cos_all[3][idd], bins = bin_type, color = colors[2], density=True, histtype = "step",label = "Walls", linewidth = 2)
plt.hist(cos_all[2][idd], bins = bin_type, color = colors[1], density=True, histtype = "step",label = "Filaments", linewidth = 2)
plt.hist(cos_all[1][idd], bins = bin_type, color = colors[0], density=True, histtype = "step",label = "Nodes", linewidth = 2)
plt.grid()
plt.legend()
plt.xlabel(r"$\cos \theta$")
plt.ylabel(rf"PDF at $z={red[idd]}$")

plt.savefig("cos_hist.png", transparent = True, bbox_inches="tight")
plt.savefig("cos_hist.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(red, mag_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(red, (np.array(mag_avg[0])-np.array(mag_err_down[0])), 
                (np.array(mag_avg[0])+np.array(mag_err_up[0])), alpha=.2, color="black")
    
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"$|v|/|v_i|$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(red, mag_avg[k], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"$|v|/|v_i|$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("mag.png", transparent = True, bbox_inches="tight")
plt.savefig("mag.pdf", transparent = True, bbox_inches="tight");

In [ ]:
tng_ages = [13.624, 13.532,
    13.433, 13.385, 13.328, 13.286, 13.256, 13.207, 13.163, 13.118, 13.071, 13.039, 
    12.960, 12.871, 12.839, 12.767, 12.691, 12.628, 12.521, 12.437, 12.337, 12.260, 
    12.115, 11.991, 11.859, 11.666, 11.585, 11.419, 11.264, 11.116, 10.961, 10.826, 
    10.674, 10.519, 10.356, 10.210, 10.059, 9.901, 9.766, 9.597, 9.470, 9.301, 
    9.147, 8.987, 8.823, 8.688, 8.514, 8.372, 8.226, 8.077, 7.925, 7.730, 
    7.610, 7.447, 7.281, 7.111, 6.981, 6.808, 6.671, 6.489, 6.345, 6.161, 
    6.017, 5.872, 5.724, 5.533, 5.371, 5.216, 5.060, 4.901, 4.741, 4.578, 
    4.414, 4.247, 4.079, 3.966, 3.764, 3.621, 3.504, 3.269, 3.149, 2.969, 
    2.787, 2.665, 2.480, 2.294, 2.169, 1.979, 1.852, 1.660, 1.466, 1.336, 
    1.140, 1.008, 0.810, 0.678, 0.475, 0.343, 0.136, 0.000
]

age = 13.803

for k in range(100):
    tng_ages[k] = age - tng_ages[k]


In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(tng_ages, diff_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(tng_ages, (np.array(diff_avg[0])-np.array(diff_err_down[0])), 
                (np.array(diff_avg[0])+np.array(diff_err_up[0])), alpha=.2, color="black")
    
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"$|v|-|v_i|$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(tng_ages, diff_avg[k], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"$|v|-|v_i|$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("diff.png", transparent = True, bbox_inches="tight")
plt.savefig("diff.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(tng_ages, mag_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(tng_ages, (np.array(mag_avg[0])-np.array(mag_err_down[0])), 
                (np.array(mag_avg[0])+np.array(mag_err_up[0])), alpha=.2, color="black")
    
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"$|v|/|v_i|$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(tng_ages, mag_avg[k], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"$|v|/|v_i|$")
ax.legend()
ax.grid()

plt.tight_layout();

# plt.savefig("diff.png", transparent = True, bbox_inches="tight")
# plt.savefig("diff.pdf", transparent = True, bbox_inches="tight");

In [ ]:
idd = -1
bin_type = "scott"
#bin_type = 10

plt.hist(mag_all[4][idd][mag_all[4][idd]<7.5], bins = bin_type, color = colors[3], density=True, histtype = "step",label = "Voids", linewidth = 2)
plt.hist(mag_all[3][idd][mag_all[3][idd]<7.5], bins = bin_type, color = colors[2], density=True, histtype = "step",label = "Walls", linewidth = 2)
plt.hist(mag_all[2][idd][mag_all[2][idd]<7.5], bins = bin_type, color = colors[1], density=True, histtype = "step",label = "Filaments", linewidth = 2)
plt.hist(mag_all[1][idd][mag_all[1][idd]<7.5], bins = bin_type, color = colors[0], density=True, histtype = "step",label = "Nodes", linewidth = 2)
plt.grid()
plt.hist(mag_all[0][idd][mag_all[0][idd]<7.5], bins = bin_type, color = "red", density=True, histtype = "step",label = "Total", linewidth = 2)
plt.legend()
plt.xlabel(r"$|v|/|v_i|$")
plt.ylabel(rf"PDF at $z={red[idd]}$")

plt.savefig("mag_hist.png", transparent = True, bbox_inches="tight")
plt.savefig("mag_hist.pdf", transparent = True, bbox_inches="tight");

In [ ]:
dens_n = []
dens_f = []
dens_w = []
dens_v = []

for i in range(n):
    print(f"Processing redshift index: {i}")
    
    dens_v.append(np.mean(norm_dens[i][voids[i] > 0]))
    dens_w.append(np.mean(norm_dens[i][walls[i] > 0]))
    dens_f.append(np.mean(norm_dens[i][filaments[i] > 0]))
    dens_n.append(np.mean(norm_dens[i][nodes[i] > 0]))

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(z_list, dens_n,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_list, dens_f,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_list, dens_w,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_list, dens_v,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"$1+\delta$")
ax.legend()
ax.set_yscale("log")

plt.savefig("nexus_dens_evol.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_dens_evol.pdf", transparent = True, bbox_inches="tight");

In [ ]:
dens_n_new = [dens_n[0]]
dens_f_new = [dens_f[0]]
dens_w_new = [dens_w[0]]
dens_v_new = [dens_v[0]]

for i in range(1,n-1):
    print(i)

    dens_v_new.append((dens_v[i-1] + dens_v[i] + dens_v[i+1])/3)
    dens_w_new.append((dens_w[i-1] + dens_w[i] + dens_w[i+1])/3)
    dens_f_new.append((dens_f[i-1] + dens_f[i] + dens_f[i+1])/3)
    dens_n_new.append((dens_n[i-1] + dens_n[i] + dens_n[i+1])/3)


dens_n_new.append(dens_n[17])
dens_f_new.append(dens_f[17])
dens_w_new.append(dens_w[17])
dens_v_new.append(dens_v[17])

In [ ]:
fig = plt.figure(figsize=(11,4))

dens_bins = np.linspace(np.min(np.log10(np.array(norm_dens).ravel())), np.max(np.log10(np.array(norm_dens).ravel())), 100) 
heatmap_data = np.zeros((len(dens_bins) - 1, 18))

for t in range(18):

    hist, _ = np.histogram(np.log10(norm_dens[t].ravel()), bins=dens_bins)
    heatmap_data[:, t] = hist

ax = fig.add_subplot(121)

red_arr=np.array(z_list)

red_centers = (red_arr[:-1] + red_arr[1:]) / 2
dens_centers = (dens_bins[:-1] + dens_bins[1:]) / 2

# Trim your data matrix to 99 columns if it's currently 100
# (or ensure it's generated as shape (9, 99))
pcm = ax.pcolormesh(
    red_centers, 
    dens_centers, 
    heatmap_data[:, :-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='magma', norm = LogNorm()
)

ax.plot(z_list, np.log10(np.mean(np.array(norm_dens), axis = (1,2,3))), color = "black", label = "Average", linewidth = 2)

ax.set_ylabel(r"$\log (1+\delta)$")
ax.set_xlabel(r"Redshift $z$")
ax.legend()
ax.grid(visible = False)

ax.set_xlim(red_arr.min(), red_arr.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Pixel Count");

ax = fig.add_subplot(122)

ax.plot(z_new, dens_n_new,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_new, dens_f_new,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_new, dens_w_new,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_new, dens_v_new,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"$1+\delta$")
ax.legend()
ax.set_yscale("log")

plt.tight_layout()

plt.savefig("nexus_dens_evol_new_new.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_dens_evol_new_new.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(12,4))

ax = fig.add_subplot(121)

ax.plot(z_new, vol_n_new,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_new, vol_f_new,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_new, vol_w_new,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_new, vol_v_new,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Volume Fraction")
ax.legend()

ax = fig.add_subplot(122)

ax.plot(z_new, mass_n_new,c=colors[0], label = "Nodes", marker = "x")
ax.plot(z_new, mass_f_new,c=colors[1], label = "Filaments", marker = "o")
ax.plot(z_new, mass_w_new,c=colors[2], label = "Walls", marker = "*")
ax.plot(z_new, mass_v_new,c=colors[3], label = "Voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Mass Fraction")
ax.legend()


plt.savefig("nexus_fractions.png", transparent = True, bbox_inches="tight")
plt.savefig("nexus_fractions.pdf", transparent = True, bbox_inches="tight");

In [ ]:
# import numpy as np
# import plotly.graph_objects as go

# # 1. Your grid (downsampled or not, it won't matter anymore)
# grid = filaments[9][::1, ::1, ::1].astype(float)
# # grid = np.pad(grid, pad_width=1, mode='constant', constant_values=0) # If using padding

# # 2. Define the physical box size
# L = 35.0 

# # 3. Create physical coordinates based on the CURRENT shape of the grid
# nx, ny, nz = grid.shape
# x = np.linspace(0, L, nx)
# y = np.linspace(0, L, ny)
# z = np.linspace(0, L, nz)

# # 4. Create the 3D coordinate grids
# X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# # 5. Plot as normal
# fig = go.Figure(data=go.Volume(
#     x=X.flatten(),
#     y=Y.flatten(),
#     z=Z.flatten(),
#     value=grid.flatten(),
#     isomin=0.5,
#     isomax=1.0,
#     opacity=0.5,
#     surface_count=1,
#     colorscale=[[0.0, 'steelblue'], [1.0, 'steelblue']],
#     showscale=False
# ))

# # Force the axes to show the physical range and keep the box cubic
# fig.update_layout(
#     scene=dict(
#         xaxis=dict(title='X [cMpc/h]', range=[0, L]),
#         yaxis=dict(title='Y [cMpc/h]', range=[0, L]),
#         zaxis=dict(title='Z [cMpc/h]', range=[0, L]),
#         aspectmode='cube' 
#     ),
#     margin=dict(l=0, r=0, b=0, t=0)
# )
# fig.show()

In [ ]:
v_z0 = voids[0] * norm_dens[0]
f_z0 = filaments[0] * norm_dens[0]
w_z0 = walls[0] * norm_dens[0]
n_z0 = nodes[0] * norm_dens[0]

v_z5 = voids[12] * norm_dens[12]
f_z5 = filaments[12] * norm_dens[12]
w_z5 = walls[12] * norm_dens[12]
n_z5 = nodes[12] * norm_dens[12]

v_z2 = voids[9] * norm_dens[9]
f_z2 = filaments[9] * norm_dens[9]
w_z2 = walls[9] * norm_dens[9]
n_z2 = nodes[9] * norm_dens[9]


v_z10 = voids[17] * norm_dens[17]
f_z10 = filaments[17] * norm_dens[17]
w_z10 = walls[17] * norm_dens[17]
n_z10 = nodes[17] * norm_dens[17]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z_min = 0.0
z_max = 12.5

# Ensure L and res are defined (e.g., L=35.0, res=128)
grid_z_min = int(np.floor((z_min / L) * res)) % res
grid_z_max = int(np.floor((z_max / L) * res)) % res

# Using plt.subplots makes managing grids and colorbars much easier!
fig, axes = plt.subplots(5, 4, figsize=(10, 12), gridspec_kw={'wspace': 0, 'hspace': 0})

data_ij = [
    [n_z0, n_z2, n_z5, n_z10],
    [f_z0, f_z2, f_z5, f_z10],
    [w_z0, w_z2, w_z5, w_z10],
    [v_z0, v_z2, v_z5, v_z10],
    [norm_dens[0], norm_dens[9], norm_dens[12], norm_dens[17]]
]

cols = ["$z = 0$", "$z = 2$", "$z = 5$", "$z = 10$"]
rows = ["Nodes", "Filaments", "Walls", "Voids","Total"]
cmaps = ["hot","viridis","inferno","bone","cubehelix"]

for i in range(5):
    
    # ---------------------------------------------------------
    # STEP 1: Pre-process all 4 redshifts to find the ROW limits
    # ---------------------------------------------------------
    row_slabs = []
    for j in range(4):
        if i == 0:
            # Nodes: full box, mean, linear scale
            slab = np.mean(data_ij[i][j], axis=2)
        else:
            # Others: sliced box, mean, log10 scale
            slab_sliced = data_ij[i][j][:, :, grid_z_min:grid_z_max] + 1
            slab = np.log10(np.mean(slab_sliced, axis=2))
            if i == 4:
                slab_sliced = data_ij[i][j][:, :, grid_z_min:grid_z_max]
                slab = np.log10(np.mean(slab_sliced, axis=2))
            
        row_slabs.append(slab)
        
    # Calculate the shared min/max across all 4 snapshots in this row
    p_min = np.percentile(row_slabs, 1)
    p_max = np.percentile(row_slabs, 99)

    # if i == 4:
    #     p_min = np.min(row_slabs)
    #     p_max = np.max(row_slabs)


    # ---------------------------------------------------------
    # STEP 2: Plot the 4 panels using the shared min/max
    # ---------------------------------------------------------
    for j in range(4):
        ax = axes[i, j]
        
        # We save the output of imshow to the variable 'im' to pass to the colorbar later
        im = ax.imshow(row_slabs[j], origin="lower", cmap=cmaps[i], 
                       extent=[0, L, 0, L], vmin=p_min, vmax=p_max)
        
        ax.set_xticks([])
        ax.set_yticks([])

        # Table Labels
        if i == 0:
            ax.set_title(cols[j], fontsize=16, pad=10)
        if j == 0:
            ax.set_ylabel(rows[i], fontsize=16, labelpad=10)

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)  
            spine.set_color('white')

    # ---------------------------------------------------------
    # STEP 3: Add one colorbar per row
    # ---------------------------------------------------------
    cbar = fig.colorbar(im, ax=axes[i, :], fraction=0.015, pad=0.01)
    
    # Add appropriate units/labels to the colorbars
    if i == 0:
        cbar.set_label(r'$1+\delta$', rotation=270, labelpad=15)
    else:
        # Reverted to your correct label!
        cbar.set_label(r'$\log(1+\delta)$', rotation=270, labelpad=15) 
        
        # Apply the correction to Rows 1, 2, 3 (and 4 if you added +1 there too)
        if i in [1, 2, 3]:
            def true_log_formatter(x, pos):
                # x is log10( (1+delta) + 1 )
                # 10**x - 1 isolates your original (1+delta)
                true_linear_val = (10**x) - 1
                
                if true_linear_val > 0:
                    # Wrapping in $ $ forces Matplotlib's math renderer, 
                    # instantly fixing the upside-down exclamation mark bug!
                    return f"${np.log10(true_linear_val):.1f}$"
                else:
                    return "$<-1$" 
            
            cbar.formatter = ticker.FuncFormatter(true_log_formatter)
            cbar.update_ticks()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z_min = 0.0
z_max = 12.5

# Ensure L and res are defined (e.g., L=35.0, res=128)
grid_z_min = int(np.floor((z_min / L) * res)) % res
grid_z_max = int(np.floor((z_max / L) * res)) % res

# Using plt.subplots makes managing grids and colorbars much easier!
fig, axes = plt.subplots(4, 4, figsize=(10, 10), gridspec_kw={'wspace': 0, 'hspace': 0})

data_ij = [
    [f_z0, f_z2, f_z5, f_z10],
    [w_z0, w_z2, w_z5, w_z10],
    [v_z0, v_z2, v_z5, v_z10],
    [norm_dens[0], norm_dens[9], norm_dens[12], norm_dens[17]]
]

cols = ["$z = 0$", "$z = 2$", "$z = 5$", "$z = 10$"]
rows = ["Filaments", "Walls", "Voids","Total"]
cmaps = ["viridis","inferno","bone","cubehelix"]

for i in range(4):
    
    # ---------------------------------------------------------
    # STEP 1: Pre-process all 4 redshifts to find the ROW limits
    # ---------------------------------------------------------
    row_slabs = []
    for j in range(4):
        if i == 5:
            # Nodes: full box, mean, linear scale
            slab = np.mean(data_ij[i][j], axis=2)
        else:
            # Others: sliced box, mean, log10 scale
            slab_sliced = data_ij[i][j][:, :, grid_z_min:grid_z_max] + 1
            slab = np.log10(np.mean(slab_sliced, axis=2))
            if i == 3:
                slab_sliced = data_ij[i][j][:, :, grid_z_min:grid_z_max]
                slab = np.log10(np.mean(slab_sliced, axis=2))
            
        row_slabs.append(slab)
        
    # Calculate the shared min/max across all 4 snapshots in this row
    p_min = np.percentile(row_slabs, 1)
    p_max = np.percentile(row_slabs, 99)

    # if i == 4:
    #     p_min = np.min(row_slabs)
    #     p_max = np.max(row_slabs)


    # ---------------------------------------------------------
    # STEP 2: Plot the 4 panels using the shared min/max
    # ---------------------------------------------------------
    for j in range(4):
        ax = axes[i, j]
        
        # We save the output of imshow to the variable 'im' to pass to the colorbar later
        im = ax.imshow(row_slabs[j], origin="lower", cmap=cmaps[i], 
                       extent=[0, L, 0, L], vmin=p_min, vmax=p_max)
        
        ax.set_xticks([])
        ax.set_yticks([])

        # Table Labels
        if i == 0:
            ax.set_title(cols[j], fontsize=16, pad=10)
        if j == 0:
            ax.set_ylabel(rows[i], fontsize=16, labelpad=10)

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)  
            spine.set_color('white')

    # ---------------------------------------------------------
    # STEP 3: Add one colorbar per row
    # ---------------------------------------------------------
    cbar = fig.colorbar(im, ax=axes[i, :], fraction=0.015, pad=0.01)
    
    # Add appropriate units/labels to the colorbars
    if i == 5:
        cbar.set_label(r'$1+\delta$', rotation=270, labelpad=15)
    else:
        # Reverted to your correct label!
        cbar.set_label(r'$\log(1+\delta)$', rotation=270, labelpad=15) 
        
        # Apply the correction to Rows 1, 2, 3 (and 4 if you added +1 there too)
        if i in [0, 1, 2]:
            def true_log_formatter(x, pos):
                # x is log10( (1+delta) + 1 )
                # 10**x - 1 isolates your original (1+delta)
                true_linear_val = (10**x) - 1
                
                if true_linear_val > 0:
                    # Wrapping in $ $ forces Matplotlib's math renderer, 
                    # instantly fixing the upside-down exclamation mark bug!
                    return f"${np.log10(true_linear_val):.1f}$"
                else:
                    return "$<-1$" 
            
            cbar.formatter = ticker.FuncFormatter(true_log_formatter)
            cbar.update_ticks()


plt.savefig("all.png", transparent=True, bbox_inches="tight")
plt.savefig("all.pdf", transparent=True, bbox_inches="tight");